# API RAG QA Pipeline

This notebook builds a RAG pipeline for the PDF-derived Markdown corpus and answers the questions in `qa_set.csv`.

It is designed for a non-local/cloud notebook environment:

- Put the converted Markdown files and `qa_set.csv` in the same folder as this notebook.
- Set `CSCS_API_KEY` in the environment, or create a file named `environment` containing `CSCS_API_KEY=...`.
- The LLM answer generation uses the Swiss AI OpenAI-compatible API.
- Extracted PDF images are captioned, embedded, retrieved, and passed to the vision model when relevant.

Retrieval settings used below:

- Chunk size: `250` words
- Chunk overlap: `50` words
- Embedding model: `Snowflake/snowflake-arctic-embed-l-v2.0`
- Cosine similarity threshold: `0.30`
- Maximum context chunks sent to the model: `6`
- Minimum fallback chunks: `3`
- Maximum context images sent to the model: `3`


## 1. Install and Import Dependencies

In [9]:
# Run this cell once in a fresh notebook environment.
%pip install -q openai python-dotenv pandas numpy tqdm


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import base64
import csv
import json
import mimetypes
import os
import re
import sqlite3
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv(dotenv_path="environment")

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR

QA_CSV = DATA_DIR / "qa_set.csv"
OUTPUT_CSV = DATA_DIR / "qa_set_api_rag_answers.csv"
DB_PATH = DATA_DIR / "rag_api_embedding_database.sqlite"

EMBEDDED_QA_CSV = """question;answer
What year where there most casualties from man-made disasters in the recorded data?;2002. In that year more than 10,000 casualties are ascribed to man-made catastrophes.
In what year did the quantity of man-made disasters peak in the recorded data between 1970 and 2023?;Man-made disasters peaked in 2005
Between 1970 and 2023 what year in the data shows the largest number of natural catastrophes?;2023.  There were 218 instances of natural catastrophes.
What year between 1994 and 2023 had the most high severity ($5 billion in damages or more) natural catastrophes?;2011 with 6.
How much higher are the 2023 insured losses than the previous 10 year average?;They are higher by 21%.
Which figure shows the trend in insured losses over data from 1994 to 2023? In this figure, what is the highest insured loss year on record?;
Before 2023, what was the highest year on record for European Severe Convective Storm losses?;
What regions does the Swiss Re report on natural catastrophes split the US into for severe convective storm risk?;
What is the highest Benefit to Cost ratio building code element described in the Swiss Re report on natural catastrophes?;
According to the Swiss Re Institute report on natural catastrophes, 2017 was a standout year in terms of insured loss damages. What were the names of the weather events which contributed most to this figure?;
Given how the Swiss Re Institute classifies primary and secondary perils, to which category can we attribute more losses in 2023? How is insurance claim tracking characterized in this category?;
In historical data from the Swiss Re Institute, which geographical grouping of countries has the smallest proportion of insured losses to uninsured losses between 2014 and 2023?;
What is the lower bound of dead or missing which have to be reported in connection to a natural catastrophe in order for the Swiss Re data to report it in their statistics?;
Excluding the overall UN average, which group of nations have more than 100 mobile broadband subscriptions per 100 residents in 2024?;
According to survey data, which feature of government web-portals experienced the largest between 2022 and 2024?;
When were the simplified four stages of E-government adoption published? Which revision of the EGDI is this related to?;
Between what years was the EGDI revision 3.0 acive?;
Given when academic articles using the term started being published, when were the terms E-government development index and Online services index introduced?;
Of the questions highlighted from the Member States Questionnaire in a chart, is there a discontinuity in the numbering of the questions featured? If so, which numbers are missing?;
What was the percentage increase over all the 193 UN member states in EGDI scores between 2022 and 2024?;
Of the countries with very high OSI levels and high EGDI divergence what is the one with the lowest Telecommunications Infrasructure Index?;
What grouping of nations has the closest OSI subindex average to the overall UN 193 average?;
Why did the average number of provided online services increase while the percentage stayed the same between 2022 and 2024? How many services were assessed in each of the 2 years?;
What percentage of european countries support filing income tax online?;
In what percentage of countries in Oceania is one able to apply for a death certificate through a fully digitized process?;
What region of the world contains the countries that enable fully digitized vehicle regitration? What is the percentage of countries in this region that support this?;
In what percentage of countries in the Americas is one able to apply for disability compensation in at least a partially online manner?;
What number of countries in Africa support digital invoicing? What is the percentage increase in that figure since 2022?;
What number of countries in Europe offer an E-procurement platform? Not percentage, number of countries.;
What fully digital service for individuals in vulnerable situations experienced the largest percentge decline between 2022 and 2024?;
What service for individuals in vulnerable situations has no fully digitized component in Oceania?;
What proportion of countries offer judiciary services in a way that is accessible on mobile or through an app?;
What landlocked countries have moved from high to very high E-government development index in the survey period?;
What small island nation moved from the middle to the high EGDI group in the last survey period?;
How has Japan named their initiative for removing bureaucratic inefficiencies and improving their digital government tools? What is the project's initial budget?;
What two departments of the UK govenrment have been merged in the effort to enhance their digital transformation?;
What country grouping appears to have the most drastic difference between EGDI levels of it's constituents?;
What is the change in percentage of cities providing information on procurement between 2022 and 2024?;
In the year before the two investment segments equalized, what was the amount invested in energy transition power generation vs fossil fuel power generation?;
How many of the top 10 costliest environmental disasters of 2023 occurred in Latin America? Which country faced the msot expensive one?;
"""

CHUNK_WORDS = 250
OVERLAP_WORDS = 50
SIMILARITY_THRESHOLD = 0.30
MAX_CONTEXT_CHUNKS = 6
MIN_CONTEXT_CHUNKS = 3
IMAGE_SIMILARITY_THRESHOLD = 0.30
MAX_CONTEXT_IMAGES = 3
IMAGE_CAPTION_MAX_TOKENS = 220
IMAGE_CAPTION_CACHE = DATA_DIR / "image_caption_cache.csv"

API_BASE_URL = "https://api.swissai.svc.cscs.ch/v1"
EMBEDDING_MODEL = "Snowflake/snowflake-arctic-embed-l-v2.0"
MODEL_NAME = "moonshotai/Kimi-K2.5-SDSC"  # Strongest model listed in the API example.
# Fast alternative: "zai-org/GLM-4.7-Flash"
# Open model alternatives: "swiss-ai/Apertus-8B-Instruct-2509", "swiss-ai/Apertus-70B-Instruct-2509"

client = OpenAI(
    base_url=API_BASE_URL,
    api_key=os.getenv("CSCS_API_KEY"),
)

if not QA_CSV.exists():
    QA_CSV.write_text(EMBEDDED_QA_CSV, encoding="utf-8")
    print("qa_set.csv was missing, so the embedded QA set was written to:", QA_CSV)

print("Working directory:", WORK_DIR)
print("Data directory:", DATA_DIR)
print("qa_set.csv exists:", QA_CSV.exists())
print("API key configured:", bool(os.getenv("CSCS_API_KEY")))

Working directory: /home/renku/work/Durham-Hackathon-2026-w2t1
Data directory: /home/renku/work/Durham-Hackathon-2026-w2t1
qa_set.csv exists: True
API key configured: True


## 2. Load Markdown Corpus

In [11]:
IMAGE_PATTERN = re.compile(r"!\[[^\]]*\]\([^)]+\)")
WORD_PATTERN = re.compile(r"\S+")


def clean_markdown(text: str) -> str:
    text = IMAGE_PATTERN.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"[ \t]+", " ", text)
    return text


markdown_files = sorted(
    path for path in DATA_DIR.glob("*.md")
    if path.name.lower() not in {"readme.md"}
)

if not markdown_files:
    raise FileNotFoundError(
        "No .md files found. Put the PDF-derived Markdown files in the same folder as this notebook."
    )

print(f"Found {len(markdown_files)} Markdown files:")
for path in markdown_files:
    print("-", path.name)

Found 4 Markdown files:
- Web Version _E-Government Survey 2024 11102024.md
- World_Inequality_Report_2026.md
- natural-catastrophe-and-climate-report-2023.md
- swissre_sigma-1_2024_english.md


## 3. Chunk the Corpus

In [12]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: int
    source: str
    chunk_index: int
    start_word: int
    end_word: int
    text: str


def chunk_text(source: str, text: str, chunk_words: int, overlap_words: int, start_id: int) -> list[Chunk]:
    if overlap_words >= chunk_words:
        raise ValueError("Overlap must be smaller than chunk size.")

    words = WORD_PATTERN.findall(clean_markdown(text))
    if not words:
        return []

    chunks = []
    step = chunk_words - overlap_words
    for chunk_index, start in enumerate(range(0, len(words), step)):
        end = min(start + chunk_words, len(words))
        chunks.append(
            Chunk(
                chunk_id=start_id + len(chunks),
                source=source,
                chunk_index=chunk_index,
                start_word=start,
                end_word=end,
                text=" ".join(words[start:end]),
            )
        )
        if end == len(words):
            break
    return chunks


chunks: list[Chunk] = []
next_id = 1
for md_path in markdown_files:
    text = md_path.read_text(encoding="utf-8", errors="replace")
    file_chunks = chunk_text(md_path.name, text, CHUNK_WORDS, OVERLAP_WORDS, next_id)
    chunks.extend(file_chunks)
    next_id += len(file_chunks)

print(f"Created {len(chunks)} chunks.")
pd.DataFrame([c.__dict__ for c in chunks[:5]])

Created 1044 chunks.


,chunk_id,source,chunk_index,start_word,end_word,text
0,1,Web Version _E-Government Survey 2024 11102024.md,0,0,250,## **E-Government Survey 2024** Accelerating D...
1,2,Web Version _E-Government Survey 2024 11102024.md,1,200,450,‘country’ and ‘economy’ as used in this Report...
2,3,Web Version _E-Government Survey 2024 11102024.md,2,400,650,"or its senior management, or of the experts wh..."
3,4,Web Version _E-Government Survey 2024 11102024.md,3,600,850,2024 UN E-GovErNmENt SUrvEy iv PREFAcE ## **Pr...
4,5,Web Version _E-Government Survey 2024 11102024.md,4,800,1050,is crucial for comprehensive digital transform...


## 4. Build API Embeddings and Save the RAG Database

In [13]:
def embed_text(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    response = client.embeddings.create(
        model=model,
        input=text,
    )
    return response.data[0].embedding


def embed_texts(texts: list[str], model: str = EMBEDDING_MODEL, batch_size: int = 16) -> np.ndarray:
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding chunks"):
        batch = texts[start : start + batch_size]
        try:
            response = client.embeddings.create(
                model=model,
                input=batch,
            )
            vectors.extend(item.embedding for item in response.data)
        except Exception:
            # Some OpenAI-compatible embedding endpoints only accept one input at a time.
            for text in batch:
                vectors.append(embed_text(text, model=model))

    matrix = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


if not os.getenv("CSCS_API_KEY"):
    raise RuntimeError("Set CSCS_API_KEY before creating API embeddings.")

chunk_texts = [chunk.text for chunk in chunks]
chunk_matrix = embed_texts(chunk_texts, batch_size=16)

print("Embedding model:", EMBEDDING_MODEL)
print("Embedding matrix shape:", chunk_matrix.shape)

Embedding chunks: 100%|██████████| 66/66 [00:08<00:00,  7.92it/s]


Embedding model: Snowflake/snowflake-arctic-embed-l-v2.0
Embedding matrix shape: (1044, 1024)


In [14]:
def save_database(db_path: Path) -> None:
    if db_path.exists():
        db_path.unlink()

    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(
            """
            CREATE TABLE metadata (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL
            );

            CREATE TABLE chunks (
                id INTEGER PRIMARY KEY,
                source TEXT NOT NULL,
                chunk_index INTEGER NOT NULL,
                start_word INTEGER NOT NULL,
                end_word INTEGER NOT NULL,
                text TEXT NOT NULL,
                embedding BLOB NOT NULL
            );

            CREATE INDEX idx_chunks_source ON chunks(source);
            """
        )
        metadata = {
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "chunk_words": CHUNK_WORDS,
            "overlap_words": OVERLAP_WORDS,
            "similarity_threshold": SIMILARITY_THRESHOLD,
            "max_context_chunks": MAX_CONTEXT_CHUNKS,
            "min_context_chunks": MIN_CONTEXT_CHUNKS,
            "embedding_backend": EMBEDDING_MODEL,
            "embedding_dimension": int(chunk_matrix.shape[1]),
            "chunk_count": len(chunks),
            "sources": sorted({chunk.source for chunk in chunks}),
        }
        conn.execute(
            "INSERT INTO metadata(key, value) VALUES (?, ?)",
            ("index", json.dumps(metadata, indent=2)),
        )
        conn.executemany(
            """
            INSERT INTO chunks(id, source, chunk_index, start_word, end_word, text, embedding)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            [
                (
                    chunk.chunk_id,
                    chunk.source,
                    chunk.chunk_index,
                    chunk.start_word,
                    chunk.end_word,
                    chunk.text,
                    chunk_matrix[row].astype(np.float32).tobytes(),
                )
                for row, chunk in enumerate(chunks)
            ],
        )
        conn.commit()
    finally:
        conn.close()


save_database(DB_PATH)
print("Saved database:", DB_PATH)

Saved database: /home/renku/work/Durham-Hackathon-2026-w2t1/rag_api_embedding_database.sqlite


## 5. Retrieval With Cosine Similarity Threshold

In [15]:
def retrieve_chunks(
    question: str,
    threshold: float = SIMILARITY_THRESHOLD,
    max_chunks: int = MAX_CONTEXT_CHUNKS,
    min_chunks: int = MIN_CONTEXT_CHUNKS,
) -> list[dict]:
    query_vector = np.asarray(embed_text(question), dtype=np.float32)
    query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-12)
    scores = chunk_matrix @ query_vector
    ranked_indices = np.argsort(scores)[::-1]

    selected = [
        {
            "chunk_id": chunks[i].chunk_id,
            "source": chunks[i].source,
            "chunk_index": chunks[i].chunk_index,
            "score": float(scores[i]),
            "text": chunks[i].text,
        }
        for i in ranked_indices
        if scores[i] >= threshold
    ][:max_chunks]

    # Fallback: if the threshold is too strict for a short/narrow question,
    # still provide the best few chunks so the model can answer or say not found.
    if len(selected) < min_chunks:
        fallback = [
            {
                "chunk_id": chunks[i].chunk_id,
                "source": chunks[i].source,
                "chunk_index": chunks[i].chunk_index,
                "score": float(scores[i]),
                "text": chunks[i].text,
            }
            for i in ranked_indices[:min_chunks]
        ]
        seen = {item["chunk_id"] for item in selected}
        selected.extend(item for item in fallback if item["chunk_id"] not in seen)

    return selected[:max_chunks]


sample_question = "How much higher are the 2023 insured losses than the previous 10 year average?"
sample_chunks = retrieve_chunks(sample_question)
[(c["score"], c["source"], c["chunk_index"]) for c in sample_chunks]

[(0.6496871709823608, 'swissre_sigma-1_2024_english.md', 12),
 (0.632580041885376, 'swissre_sigma-1_2024_english.md', 19),
 (0.6285090446472168, 'swissre_sigma-1_2024_english.md', 25),
 (0.6272820234298706, 'natural-catastrophe-and-climate-report-2023.md', 25),
 (0.6262193918228149, 'natural-catastrophe-and-climate-report-2023.md', 21),
 (0.6144300103187561, 'swissre_sigma-1_2024_english.md', 20)]

## 6. Build Image Captions, Embeddings, and Retrieval

In [ ]:
def extract_chat_message_text(message) -> str:
    """Handle OpenAI-compatible servers that do not always use message.content."""
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        return content.strip()
    if isinstance(content, list):
        pieces = []
        for item in content:
            if isinstance(item, dict):
                pieces.append(str(item.get("text") or item.get("content") or ""))
            else:
                pieces.append(str(getattr(item, "text", "") or getattr(item, "content", "")))
        joined = "\n".join(piece for piece in pieces if piece.strip()).strip()
        if joined:
            return joined

    data = message.model_dump() if hasattr(message, "model_dump") else {}
    for key in ("reasoning_content", "reasoning", "output_text", "refusal"):
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
        if isinstance(value, dict):
            text = value.get("text") or value.get("content")
            if isinstance(text, str) and text.strip():
                return text.strip()

    return "Not found in the retrieved context."


def encode_image_from_file(path: str | Path) -> str:
    path = Path(path)
    mime = mimetypes.guess_type(path.name)[0] or "image/png"
    data = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{data}"


def describe_image(image_path: str | Path, model: str = MODEL_NAME) -> str:
    encoded_image = encode_image_from_file(image_path)
    prompt = """Create a short retrieval caption for this image.

The caption is used only to decide whether this image is relevant to a user's question.
Do not transcribe the full chart or table.
Do not list every number.

Include only:
- figure/table number and title if visible
- the main topic of the image
- key variables or measures shown
- important categories such as regions, countries, years, services, sectors, or disaster types
- 1-3 notable values only if they are central to identifying the image

Keep it concise: 2-4 sentences.
If the image is decorative, a logo, or unreadable, say: Decorative or unreadable image.
"""
    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": encoded_image}},
                        ],
                    }
                ],
                temperature=0.0,
                max_tokens=IMAGE_CAPTION_MAX_TOKENS,
            )
            return extract_chat_message_text(response.choices[0].message)
        except Exception as exc:
            last_error = exc
            wait_seconds = 5 * (attempt + 1)
            print(f"Caption failed for {image_path} on attempt {attempt + 1}/3: {exc}")
            time.sleep(wait_seconds)

    return f"Image caption unavailable after API error. File: {Path(image_path).name}. Error: {last_error}"


image_files = sorted(
    list(DATA_DIR.rglob("*_images/*.png")) +
    list(DATA_DIR.rglob("*_images/*.jpg")) +
    list(DATA_DIR.rglob("*_images/*.jpeg"))
)

print("Images found:", len(image_files))
image_files[:5]

In [ ]:
def load_caption_cache(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}
    df = pd.read_csv(path, keep_default_na=False)
    if "image_path" not in df.columns or "caption" not in df.columns:
        return {}
    return dict(zip(df["image_path"], df["caption"]))


def save_caption_cache(path: Path, cache: dict[str, str]) -> None:
    pd.DataFrame(
        [{"image_path": image_path, "caption": caption} for image_path, caption in cache.items()]
    ).to_csv(path, index=False)


def build_image_records(image_paths: list[Path]) -> tuple[list[dict], np.ndarray]:
    caption_cache = load_caption_cache(IMAGE_CAPTION_CACHE)
    records = []
    embeddings = []

    for image_path in tqdm(image_paths, desc="Captioning and embedding images"):
        image_key = str(image_path)
        if image_key in caption_cache and caption_cache[image_key].strip():
            caption = caption_cache[image_key]
        else:
            caption = describe_image(image_path)
            caption_cache[image_key] = caption
            save_caption_cache(IMAGE_CAPTION_CACHE, caption_cache)

        vector = np.asarray(embed_text(caption), dtype=np.float32)
        vector = vector / max(float(np.linalg.norm(vector)), 1e-12)

        records.append(
            {
                "image_id": len(records) + 1,
                "image_path": image_key,
                "caption": caption,
            }
        )
        embeddings.append(vector)

    if embeddings:
        return records, np.vstack(embeddings).astype(np.float32)
    return records, np.empty((0, chunk_matrix.shape[1]), dtype=np.float32)


# Captioning all extracted images can take time and API budget.
# Leave as None to process all images, or set e.g. IMAGE_INDEX_LIMIT = 50 while testing.
IMAGE_INDEX_LIMIT = None
image_records, image_matrix = build_image_records(
    image_files if IMAGE_INDEX_LIMIT is None else image_files[:IMAGE_INDEX_LIMIT]
)

print("Image records:", len(image_records))
pd.DataFrame(image_records[:5])

In [ ]:
def save_image_records(db_path: Path) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.executescript(
            """
            DROP TABLE IF EXISTS images;

            CREATE TABLE images (
                id INTEGER PRIMARY KEY,
                image_path TEXT NOT NULL,
                caption TEXT NOT NULL,
                embedding BLOB NOT NULL
            );

            CREATE INDEX idx_images_path ON images(image_path);
            """
        )
        conn.executemany(
            """
            INSERT INTO images(id, image_path, caption, embedding)
            VALUES (?, ?, ?, ?)
            """,
            [
                (
                    record["image_id"],
                    record["image_path"],
                    record["caption"],
                    image_matrix[row].astype(np.float32).tobytes(),
                )
                for row, record in enumerate(image_records)
            ],
        )
        conn.commit()
    finally:
        conn.close()


save_image_records(DB_PATH)
print("Saved image records to:", DB_PATH)

In [ ]:
def retrieve_images(
    question: str,
    threshold: float = IMAGE_SIMILARITY_THRESHOLD,
    max_images: int = MAX_CONTEXT_IMAGES,
) -> list[dict]:
    if len(image_records) == 0:
        return []

    query_vector = np.asarray(embed_text(question), dtype=np.float32)
    query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-12)
    scores = image_matrix @ query_vector
    ranked_indices = np.argsort(scores)[::-1]

    selected = []
    for index in ranked_indices:
        score = float(scores[index])
        if score < threshold and len(selected) > 0:
            continue

        record = dict(image_records[int(index)])
        record["score"] = score
        selected.append(record)

        if len(selected) >= max_images:
            break

    return selected


sample_images = retrieve_images(sample_question)
[(img["score"], img["image_path"], img["caption"][:120]) for img in sample_images]

## 7. API Answer Generation

In [ ]:
SYSTEM_PROMPT = """You are a careful multimodal RAG question-answering assistant.
Use only the supplied text context, image descriptions, and attached images.
Answer concisely, preferably in one or two sentences.
First look for a direct answer. If the answer requires simple arithmetic from numbers in the context, do the calculation and say it is inferred.
When relevant, include exact years, percentages, counts, names, or figure numbers.
If the supplied text and images do not contain enough evidence, write exactly: Not found in the retrieved context.
Do not use outside knowledge. Do not invent facts.
"""


def format_context(retrieved: list[dict]) -> str:
    parts = []
    for i, chunk in enumerate(retrieved, start=1):
        text = chunk["text"]
        parts.append(
            f"[Context {i}] source={chunk['source']} chunk={chunk['chunk_index']} cosine={chunk['score']:.3f}\n{text}"
        )
    return "\n\n".join(parts)


def format_image_context(retrieved_images: list[dict]) -> str:
    if not retrieved_images:
        return "No relevant images retrieved."
    return "\n\n".join(
        f"[Image {i}] path={image['image_path']} cosine={image['score']:.3f}\n{image['caption']}"
        for i, image in enumerate(retrieved_images, start=1)
    )


def answer_question(
    question: str,
    retrieved: list[dict] | None = None,
    retrieved_images: list[dict] | None = None,
    model: str = MODEL_NAME,
) -> tuple[str, list[dict], list[dict]]:
    if retrieved is None:
        retrieved = retrieve_chunks(question)
    if retrieved_images is None:
        retrieved_images = retrieve_images(question)

    context = format_context(retrieved)
    image_context = format_image_context(retrieved_images)
    user_text = f"""Text context:
{context}

Retrieved image descriptions:
{image_context}

Question:
{question}

Answer:
"""
    content = [{"type": "text", "text": user_text}]
    for image in retrieved_images:
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": encode_image_from_file(image["image_path"])},
            }
        )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        temperature=0.0,
        max_tokens=250,
    )
    return extract_chat_message_text(response.choices[0].message), retrieved, retrieved_images

In [ ]:
# Single-question test.
# This cell requires CSCS_API_KEY to be configured.

if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before running API calls.")
else:
    evidence = retrieve_chunks(sample_question)
    image_evidence = retrieve_images(sample_question)
    answer, evidence, image_evidence = answer_question(sample_question, evidence, image_evidence)
    print("Question:", sample_question)
    print("Answer:", answer)
    print("Text evidence:")
    for item in evidence:
        print(f"- {item['source']}#{item['chunk_index']} cosine={item['score']:.3f}")
    print("Image evidence:")
    for item in image_evidence:
        print(f"- {item['image_path']} cosine={item['score']:.3f}")

## 8. Load QA CSV

In [ ]:
def read_qa_set(path: Path) -> pd.DataFrame:
    # qa_set.csv uses semicolons: question;answer
    df = pd.read_csv(path, sep=";", keep_default_na=False)
    if "question" not in df.columns:
        raise ValueError("qa_set.csv must contain a 'question' column.")
    if "answer" not in df.columns:
        df["answer"] = ""
    df["question"] = df["question"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    return df[df["question"] != ""].reset_index(drop=True)


qa_df = read_qa_set(QA_CSV)
print("Questions:", len(qa_df))
print("Existing answers:", int((qa_df["answer"] != "").sum()))
qa_df.head()

## 9. Answer All Questions

In [ ]:
def answer_qa_dataframe(
    df: pd.DataFrame,
    preserve_existing: bool = True,
    sleep_seconds: float = 0.2,
) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        question = row["question"]
        existing_answer = row["answer"]

        retrieved = retrieve_chunks(question)
        retrieved_images = retrieve_images(question)
        if preserve_existing and existing_answer:
            answer = existing_answer
        else:
            answer, retrieved, retrieved_images = answer_question(question, retrieved, retrieved_images)
            time.sleep(sleep_seconds)

        rows.append(
            {
                "question": question,
                "answer": answer,
                "retrieved_sources": " | ".join(
                    f"{item['source']}#{item['chunk_index']} ({item['score']:.3f})"
                    for item in retrieved
                ),
                "retrieved_chunk_ids": ",".join(str(item["chunk_id"]) for item in retrieved),
                "retrieved_images": " | ".join(
                    f"{item['image_path']} ({item['score']:.3f})"
                    for item in retrieved_images
                ),
                "retrieved_image_ids": ",".join(str(item["image_id"]) for item in retrieved_images),
            }
        )
    return pd.DataFrame(rows)


if not os.getenv("CSCS_API_KEY"):
    print("Set CSCS_API_KEY before answering the full QA set.")
else:
    results_df = answer_qa_dataframe(qa_df, preserve_existing=True)
    results_df.to_csv(OUTPUT_CSV, sep=";", index=False)
    print("Wrote:", OUTPUT_CSV)
    display(results_df)

## 10. Optional: Regenerate Existing Answers Too

In [ ]:
# Uncomment to regenerate every row, including rows that already had answers.
#
# results_all_regenerated = answer_qa_dataframe(qa_df, preserve_existing=False)
# results_all_regenerated.to_csv(WORK_DIR / "qa_set_api_rag_answers_regenerated.csv", sep=";", index=False)
# display(results_all_regenerated.head())

## 11. Inspect Low-Retrieval Questions

In [ ]:
# This helps tune SIMILARITY_THRESHOLD and MAX_CONTEXT_CHUNKS.
# With Snowflake/snowflake-arctic-embed-l-v2.0, start around 0.30.
# If many top scores are below 0.30, lower SIMILARITY_THRESHOLD to 0.25.
# If answers need more context, raise MAX_CONTEXT_CHUNKS to 8.

diagnostics = []
for question in qa_df["question"]:
    retrieved = retrieve_chunks(question)
    diagnostics.append(
        {
            "question": question,
            "top_score": retrieved[0]["score"] if retrieved else 0.0,
            "num_chunks_selected": len(retrieved),
            "top_source": retrieved[0]["source"] if retrieved else "",
            "top_chunk": retrieved[0]["chunk_index"] if retrieved else "",
        }
    )

diag_df = pd.DataFrame(diagnostics).sort_values("top_score")
display(diag_df.head(10))